In [36]:
# Import necessary libraries
import json
import anthropic


In [37]:
# Import environment variables from .env file
from dotenv import load_dotenv 
load_dotenv()

True

In [38]:

def add_user_messages(messages, message):
    messages.append({"role": "user", "content": message})

def add_assistant_messages(messages, message):
    messages.append({"role": "assistant", "content": message})
    
def chat(messages, model="claude-haiku-4-5", system=None, temperature=1.0, stop_sequences=[]):
    parameters = {
        "model": model,
        "max_tokens": 512,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }
    
    if system:
        parameters["system"] = system
    
    client = anthropic.Anthropic()
    response = client.messages.create(**parameters)
    return response.content[0].text

In [39]:
def run_prompt(prompt):      
    messages = []
    add_user_messages(messages, prompt)
    return chat(messages, system="Output should be without any markdown or code block, only JSON object. Don't wrap output in any markdown. Don't use any backticks.")

class PromptEvaluator:
    def __init__(self, max_concurrent_tasks=1):
        self.max_concurrent_tasks = max_concurrent_tasks

    def generate_dataset(
        self,
        task_description,
        prompt_inputs_spec={},
        output_file="dataset-prompt-techniques.json",
        num_cases=3
    ):
        messages = []
        add_user_messages(
            messages, 
            f"""
            # Goal
            Generate a dataset of unique {num_cases} plans following to description:
            <description> 
            {task_description}
            </description> 

            Each element has to contain next specs:
            <specs>
            {prompt_inputs_spec}
            </specs>

            
            # Sample:
            Sample has to be based on below json and new field named `plan` for output.
            ```json
                {prompt_inputs_spec}
            ```

            # Restrictions: 
            - The output should be a JSON an array of strings.
            - Don't wrap it in any markdown or code block.
            - Each plan should be unique and different from each other.
            - Don't propose solution. Propose plan based on the description and specs.
            """
            )    
        response = chat(messages, system="Output should be a JSON array of strings. Don't wrap it in any markdown or code block.")
        print(response)
        with open(output_file, "w") as f:
            json.dump(json.loads(response), f, indent=4)
    
    def run_evaluation(self, dataset):
        with open(dataset, "r") as f:
            tasks = json.load(f)

        results = []
        for task in tasks: 
            prompt = f"""
                # Role
                You are an expert prompt engineer and AI reviewer. 

                # Goal
                Evaluate this task and solution:
                <task>
                {task["plan"]}
                </task>
                
                Solution: 
                <output>
                {run_prompt(task["plan"])}
                </output>

                Check that plan and solution meet to specs:
                <specs>
                Height: {task["height"]}, Weight: {task["weight"]}, Goal: {task["goal"]}, Dietary restrictions: {task["restrictions"]}
                </specs>

                # Restrictions:
                - The output should be a JSON object.
                - Don't wrap it in any markdown or code block.
                - Provide your evaluation as JSON object:
                  - grade: integer from 0 to 10, where 10 is the best grade.
                  - feedback: detailed feedback on what was good and what can be improved.

            """
            grade = run_prompt(prompt)
            results.append({
                "task": task,
                "grade": grade
            })
            
        print(results)
        with open("evaluation_results.json", "w") as f:
            json.dump(results, f, indent=4)


In [40]:
evaluator = PromptEvaluator(max_concurrent_tasks=5)
evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec="""
    {
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg",
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    }
    """,
)

evaluator.run_evaluation("dataset-prompt-techniques.json")

[
  {
    "height": "180",
    "weight": "75",
    "goal": "Muscle gain and strength training",
    "restrictions": "None",
    "plan": "Breakfast: Oatmeal with banana, eggs (3), whole wheat toast, orange juice. Mid-morning: Greek yogurt with granola. Lunch: Grilled chicken breast (200g), brown rice, broccoli, olive oil. Pre-workout: Apple with almond butter. Dinner: Salmon (180g), sweet potato, asparagus, lemon dressing. Evening snack: Casein protein shake with berries."
  },
  {
    "height": "165",
    "weight": "58",
    "goal": "Weight loss and endurance",
    "restrictions": "Vegetarian",
    "plan": "Breakfast: Greek yogurt (150g), berries, flaxseeds, honey. Mid-morning: Apple with 15 almonds. Lunch: Quinoa salad with chickpeas, cucumber, tomato, tahini dressing. Pre-workout: Green tea, banana. Dinner: Lentil curry with spinach, brown rice, steamed broccoli. Evening: Herbal tea, carrot sticks."
  },
  {
    "height": "188",
    "weight": "92",
    "goal": "Power and speed improv